# CharityML: Optimizing Donor Outreach through Supervised Learning
----

## Project Overview
This project, **CharityML**, is an application of supervised learning techniques to census data to help a non-profit organization identify potential donors. The goal is to build a model that accurately predicts whether an individual makes more than $50,000 annually, the demographic most likely to donate to the cause.

### The Business Problem
CharityML is a fictional non-profit that has a list of individuals they want to reach out to for donations. However, sending mail and marketing materials to everyone is costly and inefficient. By using machine learning to filter for high-income individuals, the charity can:
* **Reduce Marketing Costs:** By avoiding individuals who are unlikely to donate.
* **Increase Conversion Rates:** By targeting the right audience with precision.
* **Maximize ROI:** Ensuring every cost spent on outreach has the highest potential return.

### Technical Approach
In this notebook, I perform an end-to-end Data Science workflow, including:
1. **Data Exploration:** Analyzing the census dataset to understand the distribution of features.
2. **Preprocessing:** Handling skewed data with Log Transformations, Scaling numerical features, and One-Hot Encoding categorical variables.
3. **Model Selection:** Comparing **Logistic Regression**, **SVM**, and **AdaBoost** to find the most efficient architecture.
4. **Optimization:** Using `GridSearchCV` to tune hyperparameters for the champion model.
5. **Feature Importance:** Extracting the most predictive signals to understand the "why" behind the results.

### Acknowledgements
This project is based on the **Finding Donors for CharityML** project from the **Udacity Data Scientist Nanodegree**. I acknowledge Udacity for the project framework, instructional context, and evaluation structure that guided this analysis.

----
## Exploring the Data

### Initial Data Assessment and Exploration

The first step in our predictive modeling pipeline is to perform a thorough exploration of the census dataset. This phase is critical for understanding the underlying distributions, identifying potential data quality issues, and gaining a baseline intuition for the features that correlate with income levels.

In [ ]:
# Import libraries necessary for this project
import numpy as np
import pandas as pd
from time import time
from IPython.display import display  # Allows the use of display() for DataFrames
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Import supplementary visualization code visuals.py
import visuals as vs

# Pretty display for notebooks
%matplotlib inline

# Load the Census dataset
data = pd.read_csv("census.csv")

# Success - Display the head records
display(data.head(n=5))

### Data Exploration
This section quantifies class balance and establishes baseline dataset characteristics that directly impact model design and evaluation. Specifically, we compute:
- Total number of records (`n_records`)
- Number of individuals earning more than $50,000 (`n_greater_50k`)
- Number of individuals earning at most $50,000 (`n_at_most_50k`)
- Positive-class prevalence (`greater_percent`)

These metrics provide the foundation for interpreting performance trade-offs in later modeling steps.

In [ ]:
# Total number of records
n_records = data.shape[0]

# Records where income is more than $50,000
n_greater_50k = data[data["income"] == ">50K"].shape[0]

# Records where income is at most $50,000
n_at_most_50k = data[data["income"] == "<=50K"].shape[0]

# Positive-class percentage in the dataset
greater_percent = (n_greater_50k / n_records) * 100

# Print the results
print("Total number of records: {}".format(n_records))
print("Individuals making more than $50,000: {}".format(n_greater_50k))
print("Individuals making at most $50,000: {}".format(n_at_most_50k))
print("Percentage of individuals making more than $50,000: {}%".format(greater_percent))

**Featureset Exploration**

* **age**: continuous. 
* **workclass**: Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked. 
* **education**: Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool. 
* **education-num**: continuous. 
* **marital-status**: Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse. 
* **occupation**: Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces. 
* **relationship**: Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried. 
* **race**: Black, White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other. 
* **sex**: Female, Male. 
* **capital-gain**: continuous. 
* **capital-loss**: continuous. 
* **hours-per-week**: continuous. 
* **native-country**: United-States, Cambodia, England, Puerto-Rico, Canada, Germany, Outlying-US(Guam-USVI-etc), India, Japan, Greece, South, China, Cuba, Iran, Honduras, Philippines, Italy, Poland, Jamaica, Vietnam, Mexico, Portugal, Ireland, France, Dominican-Republic, Laos, Ecuador, Taiwan, Haiti, Columbia, Hungary, Guatemala, Nicaragua, Scotland, Thailand, Yugoslavia, El-Salvador, Trinadad&Tobago, Peru, Hong, Holand-Netherlands.

----
## Preparing the Data
Before data can be used as input for machine learning algorithms, it often must be cleaned, formatted, and restructured. Fortunately, for this dataset, there are no invalid or missing entries we must deal with, however, there are some qualities about certain features that must be adjusted. This preprocessing can help tremendously with the outcome and predictive power of nearly all learning algorithms.

### Transforming Skewed Continuous Features
Some continuous variables are heavily right-skewed and contain extreme values that can dominate training dynamics. In this dataset, `capital-gain` and `capital-loss` are the primary examples.

Objective: visualize their distributions to confirm skewness before applying a transformation.

In [ ]:
# Split the data into features and target label
income_raw = data["income"]
features_raw = data.drop("income", axis=1)

# Visualize skewed continuous features of original data
vs.distribution(data)

For highly skewed distributions such as `capital-gain` and `capital-loss`, a logarithmic transformation is used to compress extreme ranges and stabilize model learning. Because $\log(0)$ is undefined, values are shifted by 1 before transformation.

Objective: compare pre- and post-log distributions to verify improved scale behavior and reduced outlier influence.

In [ ]:
# Log-transform the skewed features
skewed = ["capital-gain", "capital-loss"]
features_log_transformed = pd.DataFrame(data=features_raw)
features_log_transformed[skewed] = features_raw[skewed].apply(lambda x: np.log(x + 1))

# Visualize the new log distributions
vs.distribution(features_log_transformed, transformed=True)

### Normalizing Numerical Features
After log transformation, numerical features are normalized to a shared range so no single variable disproportionately influences model fitting due to scale alone.

Objective: apply Min-Max scaling to key continuous features and ensure consistent feature magnitude across the training pipeline.

In [ ]:
# Import sklearn.preprocessing.StandardScaler
from sklearn.preprocessing import MinMaxScaler

# Initialize a scaler, then apply it to the features
scaler = MinMaxScaler()  # default=(0, 1)
numerical = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]

features_log_minmax_transform = pd.DataFrame(data=features_log_transformed)
features_log_minmax_transform[numerical] = scaler.fit_transform(
    features_log_transformed[numerical]
)

# Show an example of a record with scaling applied
display(features_log_minmax_transform.head(n=5))

### Data Preprocessing

From the table in **Exploring the Data** above, we can see there are several features for each record that are non-numeric. Typically, learning algorithms expect input to be numeric, which requires that non-numeric features (called *categorical variables*) be converted. One popular way to convert categorical variables is by using the **one-hot encoding** scheme. One-hot encoding creates a _"dummy"_ variable for each possible category of each non-numeric feature. For example, assume `someFeature` has three possible entries: `A`, `B`, or `C`:

|   | someFeature |                    
| :-: | :-: |                            
| 0 |  B  |  
| 1 |  C  |
| 2 |  A  |  

We then encode this feature into `someFeature_A`, `someFeature_B` and `someFeature_C`:

|| someFeature_A | someFeature_B | someFeature_C |
| :-: | :-: | :-: | :-: |
|0| 0 | 1 | 0 |
|1| 0 | 0 | 1 |
|2| 1 | 0 | 0 |

Additionally, as with the non-numeric features, we need to convert the non-numeric target label, `income` to numerical values for the learning algorithm to work. Since there are only two possible categories for this label ("<=50K" and ">50K"), we can avoid using one-hot encoding and simply encode these two categories as 0 and 1, respectively.

In [ ]:
# One-hot encode categorical features
features_final = pd.get_dummies(features_log_minmax_transform)

# Encode target labels to numeric values
income = income_raw.map({"<=50K": 0, ">50K": 1})

# Print the number of features after one-hot encoding
encoded = list(features_final.columns)
print("{} total features after one-hot encoding.".format(len(encoded)))

# Uncomment the following line to see the encoded feature names
print(encoded)

### Shuffle and Split Data
Now all _categorical variables_ have been converted into numerical features, and all numerical features have been normalized. As always, we will now split the data (both features and their labels) into training and test sets. 80% of the data will be used for training and 20% for testing.

In [ ]:
# Import train_test_split
from sklearn.model_selection import train_test_split

# Split the 'features' and 'income' data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    features_final, income, test_size=0.2, random_state=0
)

# Show the results of the split
print("Training set has {} samples.".format(X_train.shape[0]))
print("Testing set has {} samples.".format(X_test.shape[0]))

----
## Evaluating Model Performance
In this section, we will investigate four different algorithms, and determine which is best at modeling the data. Three of these algorithms will be supervised learners of your choice, and the fourth algorithm is known as a *naive predictor*.

### Metrics and the Naive Predictor
*CharityML*, equipped with their research, knows individuals that make more than $50,000 are most likely to donate to their charity. Because of this, *CharityML* is particularly interested in predicting who makes more than $50,000 accurately. It would seem that using **accuracy** as a metric for evaluating a particular model's performace would be appropriate. Additionally, identifying someone that *does not* make more than $50,000 as someone who does would be detrimental to *CharityML*, since they are looking to find individuals willing to donate. Therefore, a model's ability to precisely predict those that make more than $50,000 is *more important* than the model's ability to **recall** those individuals. We can use **F-beta score** as a metric that considers both precision and recall:

$$ F_{\beta} = (1 + \beta^2) \cdot \frac{\text{precision} \cdot \text{recall}}{\left( \beta^2 \cdot \text{precision} \right) + \text{recall}} $$

In particular, when $\beta = 0.5$, more emphasis is placed on precision. This is called the $F_{0.5}$ score (or F-score for simplicity).

Looking at the distribution of classes (those who make at most $50,000, and those who make more), it's clear most individuals do not make more than $50,000. This can greatly affect **accuracy**, since we could simply say *"this person does not make more than $50,000"* and generally be right, without ever looking at the data! Making such a statement would be called **naive**, since we have not considered any information to substantiate the claim. It is always important to consider the *naive prediction* for your data, to help establish a benchmark for whether a model is performing well. That been said, using that prediction would be pointless: If we predicted all people made less than $50,000, *CharityML* would identify no one as donors. 


#### Note: Recap of accuracy, precision, recall

**Accuracy** measures how often the classifier makes the correct prediction. It’s the ratio of the number of correct predictions to the total number of predictions (the number of test data points).

**Precision** tells us what proportion of messages we classified as spam, actually were spam.
It is a ratio of true positives (words classified as spam, and which are actually spam) to all positives (all words classified as spam, irrespective of whether that was the correct classificatio), in other words it is the ratio of

$$\text{Precision} = \frac{\text{True Positives}} {\text{True Positives} + \text{False Positives}}$$

**Recall (sensitivity)** tells us what proportion of messages that actually were spam were classified by us as spam.
It is a ratio of true positives (words classified as spam, and which are actually spam) to all the words that were actually spam, in other words it is the ratio of

$$\text{Recall} = \frac{\text{True Positives}} {\text{True Positives} + \text{False Negatives}}$$

For classification problems that are skewed in their classification distributions, like in our case, for example, if we had 100 text messages and only 2 were spam and the remaining 98 weren't, accuracy is not a very good metric. We could classify 90 messages as not spam (including the 2 that were spam, but we classify them as not spam, hence they would be false negatives) and 10 as spam (all 10 false positives) and still get a reasonably good accuracy score. For such cases, precision and recall come in very handy. These two metrics can be combined to get the F1 score and the weighted average (harmonic mean) of the precision and recall scores. This score can range from 0 to 1, with 1 being the best possible F1 score(we take the harmonic mean when dealing with ratios).

### Naive Predictor Benchmark
* If we chose a model that always predicted an individual made more than $50,000, what would  that model's accuracy and F-score be on this dataset? You must use the code cell below and assign your results to `accuracy` and `fscore` to be used later.

**Please note** that the the purpose of generating a naive predictor is simply to show what a base model without any intelligence would look like. In the real world, ideally your base model would be either the results of a previous model or could be based on a research paper upon which you are looking to improve. When there is no benchmark model set, getting a result better than random choice is a place you could start from.



* When we have a model that always predicts 1 (i.e., the individual makes more than 50k) then our model will have no True Negatives (TN) or False Negatives (FN) as we are not making any negative (0 value) predictions. Therefore our Accuracy in this case becomes the same as our Precision (True Positives / (True Positives + False Positives)) as every prediction that we have made with value 1 that should have 0 becomes a False Positive; therefore our denominator in this case is the total number of records we have in total. 
* Our Recall score (True Positives / (True Positives + False Negatives)) in this setting becomes 1 as we have no False Negatives.

In [ ]:
TP = np.sum(
    income
)  # Counting the ones as this is the naive case. Note that 'income' is the 'income_raw' data encoded to numerical values done in the data preprocessing step.
FP = income.count() - TP  # Specific to the naive case

TN = 0  # No predicted negatives in the naive case
FN = 0  # No predicted negatives in the naive case

# Baseline classification metrics
accuracy = TP / income.count()  # = precision
recall = TP / (TP + FN)  # = 1
precision = TP / (TP + FP)

# F-score with beta = 0.5
beta = 0.5
fscore = ((1 + beta**2) * precision * recall) / ((beta**2) * precision + recall)

# Print the results
print(
    "Naive Predictor: [Accuracy score: {:.4f}, F-score: {:.4f}]".format(
        accuracy, fscore
    )
)

###  Supervised Learning Models
**The following are some of the optional supervised learning models that are currently available in** [scikit-learn](http://scikit-learn.org/stable/supervised_learning.html) **:**
- Gaussian Naive Bayes (GaussianNB)
- Decision Trees
- Ensemble Methods (Bagging, AdaBoost, Random Forest, Gradient Boosting)
- K-Nearest Neighbors (KNeighbors)
- Stochastic Gradient Descent Classifier (SGDC)
- Support Vector Machines (SVM)
- Logistic Regression

### Model Application Rationale
We will choose three of candidates models, I made sure that these models are from different architectures. Below we will discuss:

- One real-world application in industry where the model can be applied. 
- What are the strengths of the model; when does it perform well?
- What are the weaknesses of the model; when does it perform poorly?
- What makes this model a good candidate for the problem?



### 1. Logistic Regression
* **Real-world application:** Logistic Regression is widely used in clinical prediction modeling, such as systems predicting the probability of strokes or heart disease based on patient electronic health records [1].
* **Strengths of the model:** It is extremely lightweight, computationally fast to train, and offers high interpretability. Decision-makers can easily understand the weight of each feature. It is highly reliable for datasets without excessive noise [1].
* **Weaknesses of the model:** It strictly assumes a linear relationship between features and the target variable. Therefore, it struggles significantly with complex, non-linear data interactions [1].
* **Why it is a good candidate:** After the one-hot encoding step, our census data became very wide and sparse. Linear models act as a fast, reliable baseline for sparse data.

### 2. Support Vector Machines (SVM)
* **Real-world application:** SVMs are heavily utilized in bioinformatics, specifically for cancer tissue classification from microarray data and analyzing polymer properties, where the number of features far exceeds the number of samples [2].
* **Strengths of the model:** It has exceptional capability in drawing complex, non-linear decision boundaries using the Kernel Trick. It shines in high-dimensional datasets because it relies solely on support vectors rather than the entire dataset [2].
* **Weaknesses of the model:** It is computationally expensive; training is notoriously slow on large datasets. It is also highly sensitive to unscaled data and requires meticulous hyperparameter tuning [2].
* **Why it is a good candidate:** With over 100 features post-preprocessing, SVM is mathematically designed to handle this high-dimensional space without immediately overfitting.

### 3. AdaBoost (Ensemble Methods)
* **Real-world application:** AdaBoost is used in advanced medical screening for early diagnosis, such as identifying lung cancer biomarkers via electronic nose (eNose) sensors, and in complex industrial fault detection [3].
* **Strengths of the model:** It possesses a powerful error-correcting capability. By sequentially building weak learners and heavily weighting previously misclassified samples, it is incredibly effective at handling imbalanced datasets [3].
* **Weaknesses of the model:** Its strict focus on correcting past errors makes it highly sensitive to noisy data and outliers. It can waste significant computational effort trying to correctly classify anomalies that are actually just noise [3].
* **Why it is a good candidate:** The primary goal is finding potential donors, who represent a minority class (~25% of the data). AdaBoost's adaptive weighting mechanism makes it the strongest candidate for catching this specific demographic and maximizing the F-beta score.

---
**References:**
> [1] Collins, G. S., et al. (2024). "Beyond Comparing Machine Learning and Logistic Regression in Clinical Prediction Modelling: Shifting from Model Debate to Data Quality." *PMC*.
> 
> [2] Jiang, Y., et al. (2025). "Support Vector Machines in Polymer Science: A Review." *PMC*.
> 
> [3] Wang, Y., et al. (2023). "An improved AdaBoost algorithm for identification of lung cancer based on electronic nose." *PMC*.

### Training and Prediction Pipeline
To properly evaluate the performance of each model you've chosen, it's important that we create a training and predicting pipeline that allows you to quickly and effectively train models using various sizes of training data and perform predictions on the testing data. 

In [ ]:
# Evaluation metrics
from sklearn.metrics import fbeta_score, accuracy_score


def train_predict(learner, sample_size, X_train, y_train, X_test, y_test):
    """
    inputs:
       - learner: the learning algorithm to be trained and predicted on
       - sample_size: the size of samples (number) to be drawn from training set
       - X_train: features training set
       - y_train: income training set
       - X_test: features testing set
       - y_test: income testing set
    """

    results = {}

    # Fit model on sampled training data
    start = time()  # Get start time
    learner = learner.fit(X_train[:sample_size], y_train[:sample_size])
    end = time()  # Get end time

    # Record training runtime
    results["train_time"] = end - start

    # Predict on test set and training subset
    #       then get predictions on the first 300 training samples(X_train) using .predict()
    start = time()  # Get start time
    predictions_test = learner.predict(X_test)
    predictions_train = learner.predict(X_train[:300])
    end = time()  # Get end time

    # Record inference runtime
    results["pred_time"] = end - start

    # Accuracy on first 300 training samples
    results["acc_train"] = accuracy_score(y_train[:300], predictions_train)

    # Accuracy on held-out test set
    results["acc_test"] = accuracy_score(y_test, predictions_test)

    # F-score on first 300 training samples
    results["f_train"] = fbeta_score(y_train[:300], predictions_train, beta=0.5)

    # F-score on held-out test set
    results["f_test"] = fbeta_score(y_test, predictions_test, beta=0.5)

    # Success
    print("{} trained on {} samples.".format(learner.__class__.__name__, sample_size))

    # Return the results
    return results

### Initial Model Evaluation
In the code cell, you will need to implement the following:
- Import our three discussed supervised learning models.
- Initialize the three models and store them in `clf_A`, `clf_B`, and `clf_C`.
  - **Note:** We will use the default settings for each model since we will tune one specific model in a later section.
- Calculate the number of records equal to 1%, 10%, and 100% of the training data.
  - This will help us evaluate our models from different Points-Of-View.

**Note:** The following implementation may take some time to run!

In [ ]:
# Candidate supervised learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.svm import SVC

# Initialize candidate models
clf_A = LogisticRegression(random_state=42)
clf_B = AdaBoostClassifier(random_state=42)
clf_C = SVC(random_state=42)

# Sample sizes for 1%, 10%, and 100% of training data

samples_100 = len(y_train)
samples_10 = int(len(y_train) * 0.10)
samples_1 = int(len(y_train) * 0.01)

# Collect results on the learners
results = {}
for clf in [clf_A, clf_B, clf_C]:
    clf_name = clf.__class__.__name__
    results[clf_name] = {}
    for i, samples in enumerate([samples_1, samples_10, samples_100]):
        results[clf_name][i] = train_predict(
            clf, samples, X_train, y_train, X_test, y_test
        )

# Run metrics visualization for the three supervised learning models chosen
vs.evaluate(results, accuracy, fscore)

### Best Model Selection

Based on the evaluation results, the most appropriate model for identifying potential donors for CharityML is the AdaBoost Classifier. When looking at the performance on 100% of the testing data, AdaBoost achieved the highest Accuracy and the highest F-score among the three tested models. Maximizing the F-score (specifically the $F_{0.5}$ score) is critical for this project because it heavily weights precision, ensuring that the charity does not waste its limited marketing budget sending mail to individuals who are unlikely to donate.Furthermore, AdaBoost is highly efficient. While the Support Vector Classifier (SVC) was computationally expensive taking over 70 seconds to train and nearly 40 seconds to predict, AdaBoost completed both tasks in just a few seconds. This low latency is essential for future web deployment or if the charity acquires millions of new census records. Finally, AdaBoost is algorithmically well-suited for this specific problem. Our dataset is heavily imbalanced, with only about 25% of individuals making more than $50,000. AdaBoost handles this gracefully by sequentially building simple decision trees, actively penalizing itself for misclassifying the "hard-to-find" minority class, which forces it to accurately isolate our true potential donors.

### Model Explanation in Business Terms

To understand how our chosen model, **AdaBoost** (Adaptive Boosting), works, imagine you are assembling a committee of advisors to help CharityML find potential donors. During the training phase, the algorithm starts by looking at our census data and creating one very simple rule of thumb—for example, "If the person has a Master's degree, they are likely a donor." Naturally, this single, simple rule will make a lot of mistakes. 

AdaBoost then acts like a strict manager: it looks at the exact individuals the first rule misclassified, makes those specific people "heavier" or more important, and creates a second simple rule completely focused on correctly identifying *those specific individuals*. It repeats this process sequentially hundreds of times. Each new rule is built specifically to correct the mistakes made by the combination of all the rules that came before it. 

When it is time to make a prediction on a completely new person, the model does not rely on one massive, complex equation. Instead, it holds a "vote" among all the simple rules it created during training. However, not all votes are equal. The rules that proved to be highly accurate during the training phase are given a much louder voice (a higher weight) in the final decision, while the rules that struggled are given less influence. By combining the weighted votes of hundreds of specialized, error-correcting rules, AdaBoost forms one incredibly strong, unified prediction about whether an individual makes more than $50,000.

**References:**
* Schapire, R. E. (2013). *Explaining AdaBoost*. In Empirical Inference (pp. 37-52). Springer, Berlin, Heidelberg.

### Model Tuning
Next, we are going to fine tune the chosen model. Use grid search (`GridSearchCV`).

In [ ]:
# Grid-search utilities and scoring helpers
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer

# Initialize classifier
clf = AdaBoostClassifier(random_state=42)

# Hyperparameter search space

parameters = {
    "n_estimators": [x * 50 for x in range(1, 5)],
    "learning_rate": [0.1, 0.5, 1, 1.5],
}

# Custom F-beta scorer
scorer = make_scorer(fbeta_score, beta=0.5)

# Configure grid search
grid_obj = GridSearchCV(clf, parameters, scoring=scorer, n_jobs=-1)

# Fit grid search and retrieve best estimator
grid_fit = grid_obj.fit(X_train, y_train)

# Get the estimator
best_clf = grid_fit.best_estimator_

# Make predictions using the unoptimized and model
predictions = (clf.fit(X_train, y_train)).predict(X_test)
best_predictions = best_clf.predict(X_test)

# Report the before-and-afterscores
print("Unoptimized model\n------")
print(
    "Accuracy score on testing data: {:.4f}".format(accuracy_score(y_test, predictions))
)
print(
    "F-score on testing data: {:.4f}".format(fbeta_score(y_test, predictions, beta=0.5))
)
print("\nOptimized Model\n------")
print(
    "Final accuracy score on the testing data: {:.4f}".format(
        accuracy_score(y_test, best_predictions)
    )
)
print(
    "Final F-score on the testing data: {:.4f}".format(
        fbeta_score(y_test, best_predictions, beta=0.5)
    )
)

### Final Model Evaluation


#### Results:

|     Metric     | Unoptimized Model | Optimized Model |
| :------------: | :---------------: | :-------------: | 
| Accuracy Score |  0.8483                 |   0.8568              |
| F-score        |  0.7029                 |   0.7223       |



#### 1. Final Model Performance
* **Optimized Model Accuracy:** 0.8568
* **Optimized Model F-score:** 0.7223

#### 2. Comparison to Unoptimized Model
The optimized model performed **better** than the unoptimized model. While the accuracy saw a modest increase of approximately **0.85%** (from 0.8483 to 0.8568), the F-score saw a more meaningful improvement of nearly **2%** (from 0.7029 to 0.7223). This improvement in the F-score is particularly valuable for CharityML, as it indicates that the hyperparameter tuning successfully sharpened the model's ability to prioritize precision, ensuring we are more certain about the donors we identify and reducing the cost of wasted marketing.

#### 3. Comparison to Naive Predictor
The results from the optimized model are **significantly superior** to the naive predictor benchmarks established in Question 1. 
* **Accuracy:** Increased from **0.2478** (Naive) to **0.8568** (Optimized).
* **F-score:** Increased from **0.2917** (Naive) to **0.7223** (Optimized).

The Naive Predictor essentially guessed "Yes" for every individual, leading to a high rate of false positives. Our final AdaBoost model has learned complex socio-economic patterns within the census data, providing a prediction that is over **2.4 times more effective** at identifying the correct donors while correctly ignoring non-donors.

----
## Feature Importance

An important task when performing supervised learning on a big number of features dataset like our current dataset now (> 100) is determining which features provide the most predictive power. By focusing on the relationship between only a few crucial features and the target label we simplify our understanding of the phenomenon, which is most always a useful thing to do. In the case of this project, that means we wish to identify a small number of features that most strongly predict whether an individual makes at most or more than $50,000.

### Extracting Feature Importance
We are going to use `feature_importance_` attribute. This attribute is a function that ranks the importance of each feature when making predictions based on the chosen algorithm.

In [ ]:
# Model with feature_importances_ support
from sklearn.ensemble import AdaBoostClassifier

# Train model on full training set
model = AdaBoostClassifier().fit(X_train, y_train)

# Extract ranked feature importances
importances = model.feature_importances_

# Plot
vs.feature_plot(importances, X_train, y_train)

### Interpreting Extracted Feature Importance

The model has mathematically determined that out of all available columns, these five carry the strongest signal for high income:

1. **Established Wealth & Household Stability (capital-gain & marital-status_Married-civ-spouse)**: Together, these two features carry over 50% of the model's total predictive weight. Large capital gains are a direct mathematical signal of active, profitable investments and high disposable income. When paired with being married to a civilian spouse—which heavily correlates with older, established, and often dual-income households the model finds its strongest demographic for potential donors.

2. **education-num**: The total years of education. This acts as a logical gatekeeper for higher-paying salaried positions in the workforce.

3. **age**: Income naturally scales with age, seniority, and career experience.

4. **capital-loss**: Similar to capital gains, reporting capital losses indicates the individual is actively participating in financial markets and holds investment portfolios, which implies a higher baseline of wealth.

### Feature Selection
How does a model perform if we only use a subset of all the available features in the data? With less features required to train, the expectation is that training and prediction time is much lower at the cost of performance metrics.

In [ ]:
# Import functionality for cloning a model
from sklearn.base import clone

# Reduce the feature space
X_train_reduced = X_train[X_train.columns.values[(np.argsort(importances)[::-1])[:5]]]
X_test_reduced = X_test[X_test.columns.values[(np.argsort(importances)[::-1])[:5]]]

# Train on the "best" model found from grid search earlier
clf = (clone(best_clf)).fit(X_train_reduced, y_train)

# Make new predictions
reduced_predictions = clf.predict(X_test_reduced)

# Report scores from the final model using both versions of data
print("Final Model trained on full data\n------")
print(
    "Accuracy on testing data: {:.4f}".format(accuracy_score(y_test, best_predictions))
)
print(
    "F-score on testing data: {:.4f}".format(
        fbeta_score(y_test, best_predictions, beta=0.5)
    )
)
print("\nFinal Model trained on reduced data\n------")
print(
    "Accuracy on testing data: {:.4f}".format(
        accuracy_score(y_test, reduced_predictions)
    )
)
print(
    "F-score on testing data: {:.4f}".format(
        fbeta_score(y_test, reduced_predictions, beta=0.5)
    )
)

### Effects of Feature Selection

**Comparison of Metrics:**

The final model's performance on the reduced data (using only 5 features) is very slightly lower than when trained on the full dataset. The accuracy dropped by merely ~0.82% (from 0.8568 to 0.8486), and the F-score dropped by ~1.18% (from 0.7223 to 0.7105). Given that we eliminated over 95% of the input variables (from 103 one-hot encoded features down to just 5), the model retained an exceptional amount of its predictive power. This proves that these 5 features carry the vast majority of the "signal" in the data.

**Consideration of Training Time:**

If training time or prediction latency were significant factors, I would absolutely use the reduced data as the training set. When architecting machine learning models for web deployment and high-volume production, holding 100+ features in memory and computing them for millions of incoming user requests is computationally expensive and slow. Sacrificing roughly 1% of the F-score to gain a massive reduction in computational overhead, storage constraints, and training time is a highly pragmatic and standard trade-off for building scalable systems.